# Spike Data Playground
This notebook achieves the following:
1. Download real-world raw unlabeled MEA data from the International Brain Lab using an API
2. Simulate raw synthetic MEA data
3. Perform filtering

In [5]:
# update pip, then auto-install any missing packages by PyPI name
import sys, subprocess, re
from importlib import metadata as im

pkgs = [
    'ipykernel', 'numpy', 'pandas', 'matplotlib', 'scipy',
    'ONE-api', 'spikeinterface[full]', 'mtscomp',
    'MEArec[templates]', 'probeinterface'
]

def _pip(*args):
    return subprocess.run([sys.executable, '-m', 'pip', *args], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0

def _has(dist_name: str) -> bool:
    try:
        im.version(dist_name)  # checks distribution by PyPI/project name
        return True
    except im.PackageNotFoundError:
        return False

# 1) Ensure pip is current
_pip('install', '--upgrade', 'pip', '--quiet')

# 2) Ensure each package (extras like [full] are fine for install; strip for presence check)
report = {}
for spec in pkgs:
    dist = re.split(r'[\[\]]', spec)[0]          # e.g., 'spikeinterface[full]' -> 'spikeinterface'
    if _has(dist):
        report[spec] = 'already installed'
    else:
        ok = _pip('install', spec, '--quiet')
        report[spec] = 'installed' if ok and _has(dist) else 'install failed'

# 3) Summary
w = max(len(k) for k in report) + 2
for k, v in report.items(): print(f"{k.ljust(w)}{v}")

ipykernel             already installed
numpy                 already installed
pandas                already installed
matplotlib            already installed
scipy                 already installed
ONE-api               already installed
spikeinterface[full]  already installed
mtscomp               installed
MEArec[templates]     already installed
probeinterface        already installed


# Real World Data
One source for multielectrode extracellular array (MEA) data is the the International Brain Lab ([IBL](https://www.internationalbrainlab.com/data)). It is essentially a collection of labs from across the world that collect data in some standardized format (ie., all labs try to follow same protocol for collecting data).

The data we are using was collected in part of a "Brain Wide Map" initiative. **You should read the brief overview** [here](https://docs.internationalbrainlab.org/notebooks_external/2025_data_release_brainwidemap.html). Keep in mind of the following:
- The raw data can be obtained from [here](https://docs.internationalbrainlab.org/notebooks_external/data_structure.html).
- The data is organized in a particular format, **which you should read** from [here](https://docs.internationalbrainlab.org/notebooks_external/data_structure.html).
- Data is of various format, but we are primarily interested in `raw_ephys_data` (raw, electrophysiological data colelcted from Neuropixels)

We will use the [ONE API](https://int-brain-lab.github.io/iblenv/notebooks_external/data_download.html) setup to download data.

**Note: The raw data is very large, but we can download snippets!**

In [ ]:
from one.api import ONE
from pathlib import Path

# Specify download directory (used later)
DOWNLOAD_DIR = Path('IBL_data')
DOWNLOAD_DIR.mkdir(parents = True, exist_ok = True)

one = ONE(
    base_url = 'https://openalyx.internationalbrainlab.org',
    password = 'international',
    cache_dir = DOWNLOAD_DIR,
    silent = True
)

### Querying for Data from Experiment
We will use the data that was also used in [this](https://proceedings.neurips.cc/paper_files/paper/2023/file/83c637c3bc0ca88eda6cf4f5f45bdced-Paper-Conference.pdf) paper. I am quoting a snippet below:
> To train and evaluate our model, we make use of two publicly available extracellular recordings
published by the International Brain Laboratory (IBL): the DY016 and DY009 recordings [54]. These
multi-region, Neuropixels 1.0 recordings are taken from a mouse performing a decision-making task
(see Supplementary Materials Section C for more details).

We will look at the raw MEA data from `DY_016`, which can (also) be found [here](https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/). We will load in data from the experiment on date `2019-09-11` session `001`.

You can cross-check by checking the online directory here: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/.

In [ ]:
# Find the exact session (DY_016 / 2020-09-11 / 001)
eids = one.search(
    subject    = 'DY_016',
    date_range = ['2020-09-11', '2020-09-11'],
    number     = 1 # "001" session number
)  

if not eids: raise RuntimeError("Session not found.")

# Convert experiment ids into a list of strings (if not already)
if not isinstance(eids, list):
    eids = [str(eid) for eid in [eids]]

# The triplet (subject, date, session number) corresponds to an experiment.
# This is uniquely identifiable by an experiement id (or eID).
eid = eids[0]

### Number of Probes
Now, every experiment is going to have several different datasets. We only care about the collection with **raw electrophysiological data**. Depending on the number of probes (MEAs), we may have multiple sets of collections.

**Note: The raw data for each probe could be up to 50 GB! While this experiment has two probes (`00` and `01`), we will only download data for the first one!**

This experiment has two probes, as seen here: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/.

In [ ]:
# List all folders that have the format "raw_ephys_data/probe*"
collections = one.list_collections(eid, collection = 'raw_ephys_data/probe*')

print(f'# Probes: {len(collections)}')
print(f'Names: {[col.split("/")[-1] for col in collections]}')

print(collections[-1])


### Relevant Files to Download (from each probe)
For each probe, we want to download the following files (where "ap" means action potential):
- `.ap.meta` (metadata)
- `.ap.cbin` (compressed raw binary)
- `.ap.ch` (commpression header chunks)

**First, we get the relevant file paths and ids.**

Note these file suffixes are slightly different from what you will see in the online directory (but same content, the API just displays the names differetly). For example, see the following for probe `000`: https://ibl.flatironinstitute.org/public/danlab/Subjects/DY_016/2020-09-11/001/raw_ephys_data/probe00/

In [ ]:
import pandas as pd

# This will get datasets for a SINGLE probe (as a data frame)
df = one.list_datasets(eid, collection='raw_ephys_data/probe00', details=True)

# Ensure df is a DataFrame (convert if necessary)
if not isinstance(df, pd.DataFrame): df = pd.DataFrame(df)

# Files we want to download (action potential recordings related)
SUFFIXES = ('.ap.cbin', '.ap.meta', '.ap.ch')

# Keep only the AP files we care about
mask = df['rel_path'].str.endswith(SUFFIXES)
ap_df = df.loc[mask, ['rel_path', 'file_size']].copy()

# Get file size in GB
ap_df['size_GB'] = ap_df['file_size'] / 1e9

# Display only rel_path and size_GB, sorted by rel_path
ap_df[['rel_path', 'size_GB']].sort_values('rel_path')

### Downloading Files
Now, we will download all the data. Below I provide a useful function for copying/pasting. Just be sure to edit the following (if you know what you are doing):
- `DOWNLOAD_DIR`
- `SUFFIXES`
- `config`

The final function code block is removed 

In [ ]:
# This code block will be useful for copying/pasting to easily download relevant data 
import shutil

DOWNLOAD_DIR = Path('IBL_data')
DOWNLOAD_DIR.mkdir(parents = True, exist_ok = True)
SUFFIXES = ('.ap.cbin', '.ap.meta', '.ap.ch')

configs: list[dict[str, str | int | list[str]]] = []

# Can add your own
config = {
    'subject':    'DY_016',
    'date_range': ['2020-09-11', '2020-09-11'],
    'number':     1,    # Session "001"
    'all probes': False # If true, downloads all. Else, downloads first one.
}
configs.append(config)

def _move_and_cleanup(file_paths: list[Path], destination_dir: Path):
    """
    Move files to destination and clean up the original directory structure.
    
    Args:
        file_paths: List of PosixALFPath objects (or Path-like objects)
        destination_dir: Path object for destination directory
    """
    # Ensure destination exists
    destination_dir = Path(destination_dir)
    destination_dir.mkdir(parents = True, exist_ok = True)
    
    # Move each file
    for file_path in file_paths:
        dest_file = destination_dir / file_path.name
        shutil.copy2(file_path, dest_file)
        
    # Delete the downloaded directory from ONE API
    downloaded_root_dir = Path(file_paths[0].parts[0]) / file_paths[0].parts[1]
    shutil.rmtree(downloaded_root_dir)

def _download_raw_ephys_data_helper(eid, probe) -> list[Path]:
    files = one.list_datasets(eid, collection = f'raw_ephys_data/{probe}', details = False)
    
    # File to keep
    keep_files = [str(file) for file in files if any([str(file).endswith(suffix) for suffix in SUFFIXES])]
    
    keep_file_paths: list[Path] = []
    for keep_file in keep_files:
        file_path = one.load_dataset(eid, dataset = keep_file, download_only = True)
        keep_file_paths.append(Path(file_path))
    
    return keep_file_paths

def download_raw_ephys_data(configs: list[dict]):
    one = ONE(
        base_url  = 'https://openalyx.internationalbrainlab.org',
        password  = 'international',
        cache_dir = DOWNLOAD_DIR,
        silent    = True
    )
    
    for config in configs:
        # Get all eIDs
        eids = one.search(
            subject    = config['subject'],
            date_range = config['date_range'],
            number     = config['number']
        )
        if not isinstance(eids, list): eids = [str(eid) for eid in [eids]]
        
        for eid in eids:
            # Get all probes
            collections = one.list_collections(eid, collection = 'raw_ephys_data/probe*')
            probes = [col.split("/")[-1] for col in collections]
            
            # Download only first probe (which will be listed last), if specified.
            if not config['all probes']: probes = [probes[-1]]
            
            for probe in probes:
                # Download files
                file_paths = _download_raw_ephys_data_helper(eid, probe)
                
                # Move files and delete extra ones downloaded
                probe_dir = DOWNLOAD_DIR / f"{config['subject']}_{probe}"
                print(f'Moving files to {probe_dir}...')
                _move_and_cleanup(file_paths, probe_dir)
                print('Done!')
    print(f'All files downloaded and moved to {DOWNLOAD_DIR}.')

# download_raw_ephys_data(configs)

# Synthetic Data (Under Construction)
The primary packages we will rely on are:
- `probeinterface` to get the configuration of the exact Neuropixels model we want to simulate for
- `MEArec` to simuluate MEA recordings, based on some template

In [ ]:
from probeinterface import get_probe
from probeinterface.io import write_probeinterface

probe = get_probe('imec', 'NP1000')

In [ ]:
import MEArec as mr
templates_params   = mr.get_default_templates_params()
cell_models_folder = mr.get_default_cell_models_folder()

print(cell_models_folder)
print(templates_params)

# Filtering (Under Partial Construction)
TBD

In [1]:
from pathlib import Path
import numpy as np
import matplotlib.pylab as plt
import scipy.signal

import spikeinterface as si
import spikeinterface.preprocessing as spre
import spikeinterface.widgets as sw
import spikeinterface.extractors as se

# ---------- CONFIG ----------
base_path       = Path("IBL_data/DY_016_probe00")
VIEW_START_S    = 5.0
VIEW_DURATION_S = 2.0
FREQ_MIN        = 300.0
FREQ_MAX        = 6000.0
LINE_NOISE_FREQ = 60.0

In [2]:
# ---------- LOAD DATA ----------
recording = se.read_cbin_ibl(folder_path=base_path)

sf = recording.get_sampling_frequency()
num_ch = recording.get_num_channels()
duration_s = recording.get_total_duration()
print(f"Loaded IBL CBIN | freq = {sf:.1f} Hz | channels = {num_ch} | duration = {duration_s:.1f} seconds")

Loaded IBL CBIN | freq = 30000.0 Hz | channels = 384 | duration = 4164.1 seconds


In [5]:
# Get first 10 seconds of data
n_seconds = 10
n_samples = int(n_seconds * recording.get_sampling_frequency())

data = recording.get_traces(start_frame=0, end_frame=n_samples)
data = data.T  # Shape: (n_channels, n_samples)

print(f"Data shape: {data.shape}")  # (384, 300000) for 10s @ 30kHz

print(data)

print(f"Data range: [{data.min()}, {data.max()}]")
print(f"Data mean: {data.mean():.2f}")
print(f"Data std: {data.std():.2f}")

Data shape: (384, 300000)
[[-12 -19 -20 ... -34 -35 -35]
 [ 29  26  16 ...   9   2  -3]
 [  1   3   7 ...  -7  -3 -15]
 ...
 [  3   6   4 ...   5   7   4]
 [ 14   9  15 ...  -2   4   6]
 [ 10   5   8 ...   6   8   7]]
Data range: [-337, 213]
Data mean: 1.33
Data std: 21.89


In [ ]:
import numpy as np

# Get your data
n_seconds = 10
n_samples = int(n_seconds * recording.get_sampling_frequency())
data = recording.get_traces(start_frame=0, end_frame=n_samples).T

# ---------- OPTION 1: NumPy format (recommended for arrays) ----------
np.save('data_10s.npy', data)
# Load later with: data = np.load('data_10s.npy')

In [20]:
# ---------- SELECT SUBSET OF CHANNELS ----------
# Pick 16 evenly spaced channels for visualization
ch_ids = recording.channel_ids
idx = np.linspace(0, len(ch_ids) - 1, min(16, len(ch_ids)), dtype=int)
subset_channels = ch_ids[idx]

# ---------- PREPROCESSING ----------
print("\nApplying filters...")
rec_bp    = spre.bandpass_filter(recording, freq_min=FREQ_MIN, freq_max=FREQ_MAX)
rec_notch = spre.notch_filter(rec_bp, freq=LINE_NOISE_FREQ, q=30)
rec_filt  = spre.common_reference(rec_notch, reference='global', operator='median')
print("✓ Bandpass (300-6000 Hz) → Notch (60 Hz) → Common median reference")

f_raw, p_raw     = scipy.signal.welch(recording.get_traces(), fs = recording.get_sampling_frequency())
f_bp, p_bp       = scipy.signal.welch(rec_bp.get_traces(),    fs = recording.get_sampling_frequency())
f_notch, p_notch = scipy.signal.welch(rec_notch.get_traces(), fs = recording.get_sampling_frequency())
f_filt, p_filt   = scipy.signal.welch(rec_filt.get_traces(),  fs = recording.get_sampling_frequency())

fig, ax = plt.subplots()
ax.semilogy(
    f_raw, p_raw[0],
    f_bp, p_bp[0],
    f_notch, p_notch[0],
    f_filt, p_filt[0]
)


Applying filters...
✓ Bandpass (300-6000 Hz) → Notch (60 Hz) → Common median reference


: 

# Download Real World Data (as Numpy Object)

In [7]:
from pathlib import Path
import spikeinterface.extractors as se

base_path = Path("IBL_data/DY_016_probe00")

# ---------- LOAD DATA ----------
recording = se.read_cbin_ibl(folder_path=base_path)

sf = recording.get_sampling_frequency()
num_ch = recording.get_num_channels()
duration_s = recording.get_total_duration()
print(f"Loaded IBL CBIN | freq = {sf:.1f} Hz | channels = {num_ch} | duration = {duration_s:.1f} seconds")


Loaded IBL CBIN | freq = 30000.0 Hz | channels = 384 | duration = 4164.1 seconds


In [ ]:
import numpy as np

# Get your data
n_seconds = 10
n_samples = int(n_seconds * recording.get_sampling_frequency())
data = recording.get_traces(start_frame=0, end_frame=n_samples).T

# ---------- OPTION 1: NumPy format (recommended for arrays) ----------
np.save('data_10s.npy', data) # (channels x samples)

# Load later with: data = np.load('data_10s.npy')